# Constrained double pendulum

This notebook demonstrates the generic workflow in the Lagrangian machinery for holonomic systems. 

 * Decide upon your configuration space: define the Lagrangian coordinates $\mathbf q$
 * Write the Kinetic energy $T(\mathbf q, \dot{\mathbf q})$ 
 * Write the Potential energy $V(\mathbf q)$
 * Assemble the Lagrangian $L = T - V$
 * Define the holonomic constraints $\mathbf F(\mathbf q) = 0$
 * Assemble the equation of motion using quivira : `quivira._lagrangian_math.assemble_eom(L, F, q, dq)`
 
 
 The idea is to keep the problem-specific setup explicit: define the coordinates, write the Lagrangian, compute the unconstrained accelerations, assemble the constraint Jacobian, and then enforce the constraints with the helper functions from `quivira._lagrangian_math`.

The example is a double pendulum fixed at the top, released from a sideways configuration, and allowed to swing under gravity.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import heyoka as hy
from IPython.display import HTML
import quivira as qv

The generalized coordinates are the center positions and orientations of the two rods. We keep the model explicit:

$$q = (x_0, y_0, \theta_0, x_1, y_1, \theta_1)$$

and

$$\dot q = (\dot x_0, \dot y_0, \dot \theta_0, \dot x_1, \dot y_1, \dot \theta_1).$$

The first rod is pinned at the top and the second rod is connected to the first, so the geometry is encoded by algebraic constraints rather than by a built-in mechanism model.

In [ ]:
# Generalized coordinates: center positions and orientations of the two rods.
# q = [x0, y0, theta0, x1, y1, theta1]
# qd = [xd0, yd0, thetad0, xd1, yd1, thetad1]
q = list(hy.make_vars("x0", "y0", "theta0", "x1", "y1", "theta1"))
qd = list(hy.make_vars("xd0", "yd0", "thetad0", "xd1", "yd1", "thetad1"))

The Lagrangian is assembled from the kinetic and potential energies:

$$
T = \frac{1}{2} m_0 (\dot x_0^2 + \dot y_0^2)
  + \frac{1}{24} m_0 L_0^2 \dot \theta_0^2
  + \frac{1}{2} m_1 (\dot x_1^2 + \dot y_1^2)
  + \frac{1}{24} m_1 L_1^2 \dot \theta_1^2,
$$

and

$$
V = m_0 g y_0 + m_1 g y_1.
$$

So the dynamics are governed by

$$
L = T - V.
$$

In [ ]:
L0 = 1.0
L1 = 1.0
m0 = 1.0
m1 = 1.0
g = 9.81

# Kinetic and potential energies for the two rigid rods. Each rod contributes
# center-of-mass translational energy and rotational energy about its center.
T = 0.5 * m0 * (qd[0] ** 2 + qd[1] ** 2)
T += 0.5 * m0 * (L0 ** 2) * (qd[2] ** 2) / 12
T += 0.5 * m1 * (qd[3] ** 2 + qd[4] ** 2)
T += 0.5 * m1 * (L1 ** 2) * (qd[5] ** 2) / 12

V = m0 * g * q[1] + m1 * g * q[4]
L = T - V

The constraints encode the geometry of the linkage. For the first rod the center must satisfy

$$
\begin{aligned}
F_1 &= x_0 - \frac{L_0}{2}\cos\theta_0 = 0, \\
F_2 &= y_0 - \frac{L_0}{2}\sin\theta_0 = 0.
\end{aligned}
$$

For the second rod we link its center to the end of the first rod:

$$
\begin{aligned}
F_3 &= x_1 - x_0 - \frac{L_0}{2}\cos\theta_0 - \frac{L_1}{2}\cos\theta_1 = 0, \\
F_4 &= y_1 - y_0 - \frac{L_0}{2}\sin\theta_0 - \frac{L_1}{2}\sin\theta_1 = 0.
\end{aligned}
$$

In [ ]:
# Top attachment: the first rod starts at the origin and is anchored there.
# The second rod connects to the end of the first rod.
F = np.asarray([
    q[0] - 0.5 * L0 * hy.cos(q[2]),
    q[1] - 0.5 * L0 * hy.sin(q[2]),
    q[3] - q[0] - 0.5 * L0 * hy.cos(q[2]) - 0.5 * L1 * hy.cos(q[5]),
    q[4] - q[1] - 0.5 * L0 * hy.sin(q[2]) - 0.5 * L1 * hy.sin(q[5]),
], dtype=object)

The constrained equations of motion are assembled in first-order ODE form. The accelerations are adjusted to satisfy the constraints. The function returns both the ODE system and symbolic expressions for the Lagrange multipliers, which represent the constraint reactions. We can later evaluate those expressions along the simulated trajectory to see how the reactions change over time.

In [ ]:
ode_system, lambda_expr = qv.lagrange_eom(L, F, q, qd, return_multipliers=True)

We initialize the system from a non-equilibrium sideways pose, with both rods tilted away from vertical and all velocities set to zero:

$$
\theta_0 = 1.1, \qquad \theta_1 = -1.5, \qquad \dot q = 0.
$$

The remaining inital state values are calculated from those. This gives a clean starting condition: the system begins at rest and the motion is driven by gravity.

In [ ]:
# Start from a sideways double-pendulum configuration with zero initial speeds.
theta0 = 1.1
theta1 = -1.5

x0 = 0.5 * L0 * np.cos(theta0)
y0 = 0.5 * L0 * np.sin(theta0)
x1 = x0 + 0.5 * L0 * np.cos(theta0) + 0.5 * L1 * np.cos(theta1)
y1 = y0 + 0.5 * L0 * np.sin(theta0) + 0.5 * L1 * np.sin(theta1)

state0 = [x0, y0, theta0, x1, y1, theta1, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]

Heyoka constructs an adaptive Taylor integrator from the ODE system. It automatically generates and JIT-compiles the code needed to compute Taylor-series steps. This creates the integrator but does not advance the system in time; propagation happens in the later propagate_grid call.

In [ ]:
integrator = hy.taylor_adaptive(ode_system)

We set the integrator to the initial state at time zero, then propagate it across a grid of 401 time points from 0 to 8. Heyoka advances the system using adaptive Taylor steps and returns the state at each requested time; the final element of the return value is the array of sampled states.

In [ ]:
integrator.state[:] = state0
integrator.time = 0.0

t_end = 8
tgrid = np.linspace(0.0, t_end, 401)
solution = integrator.propagate_grid(tgrid)[-1]

Now lets animate our result

In [ ]:
# Draw the rods as line segments from their anchor points.
fig, ax = plt.subplots(figsize=(6, 6))
ax.set_aspect('equal')
ax.grid(True)
ax.set_xlabel('x')
ax.set_ylabel('y')
frame_extent = L0 + L1 + 0.25
ax.set_xlim(-frame_extent, frame_extent)
ax.set_ylim(-frame_extent, frame_extent)

rod0_line, = ax.plot([], [], 'o-', lw=3, color='tab:blue')
rod1_line, = ax.plot([], [], 'o-', lw=3, color='tab:orange')
trail, = ax.plot([], [], '-', lw=1, alpha=0.6, color='black')

def rod_points(state, rod_length):
    x, y, theta = state[0], state[1], state[2]
    u = np.array([np.cos(theta), np.sin(theta)])
    start = np.array([x, y]) - 0.5 * rod_length * u
    end = np.array([x, y]) + 0.5 * rod_length * u
    return start, end

def init_anim():
    rod0_line.set_data([], [])
    rod1_line.set_data([], [])
    trail.set_data([], [])
    return rod0_line, rod1_line, trail

def update_anim(frame):
    state = solution[frame]
    x0, y0, theta0, x1, y1, theta1 = state[:6]

    start0, end0 = rod_points(np.array([x0, y0, theta0]), L0)
    start1, end1 = rod_points(np.array([x1, y1, theta1]), L1)

    rod0_line.set_data([start0[0], end0[0]], [start0[1], end0[1]])
    rod1_line.set_data([start1[0], end1[0]], [start1[1], end1[1]])

    tip_x = solution[: frame + 1, 3] + 0.5 * L1 * np.cos(solution[: frame + 1, 5])
    tip_y = solution[: frame + 1, 4] + 0.5 * L1 * np.sin(solution[: frame + 1, 5])
    trail.set_data(tip_x, tip_y)

    return rod0_line, rod1_line, trail

anim = FuncAnimation(
    fig,
    update_anim,
    frames=len(solution),
    interval=20,
    blit=False,
    init_func=init_anim,
)

display(HTML(anim.to_jshtml()))
plt.close(fig)

At each saved time, we evaluate the four geometric constraint functions at the simulated coordinates. For an exact constrained motion, each residual is zero:

$$\bm F(\bm q(t)) = \bm 0.$$

The plot shows the absolute residuals, $|F_i(\bm q(t))|$, on a logarithmic scale. Since each $F_i$ measures a positional mismatch, its magnitude has units of length.

This is a numerical check that the simulated rods remain attached at the top pin and hinge. Residuals close to zero indicate that the constraints are being maintained; growing residuals indicate geometric drift.

In [ ]:
# Evaluate the position-constraint residuals at every sampled state.
constraint_func = hy.cfunc(list(F), vars=q)
constraint_inputs = np.ascontiguousarray(solution[:, :6].T)
constraint_values = constraint_func(constraint_inputs).T

fig, ax = plt.subplots(figsize=(8, 4))
for index in range(constraint_values.shape[1]):
    ax.semilogy(
        tgrid,
        np.maximum(np.abs(constraint_values[:, index]), np.finfo(float).tiny),
        label=f"$|F_{index + 1}|$",
    )

ax.set_xlabel("Time")
ax.set_ylabel("Absolute constraint residual")
ax.set_title("Double-pendulum constraint drift")
ax.grid(True, which="both")
ax.legend()
plt.ylim(1e-20, 1e-6)
plt.show()

The Lagrange multipliers give the reaction components associated with the constraints. For the top pin, the reaction on rod 0 is $(\lambda_1, \lambda_2)$. At the hinge, the reaction on rod 1 is $(\lambda_3, \lambda_4)$, while the reaction on rod 0 is the opposite vector.

Each curve shows the magnitude of the joint reaction vector. For example, the top-pin magnitude is $\sqrt{\lambda_1^2 + \lambda_2^2}$. These forces show how the supports and hinge load the idealized rigid rods during motion, which is useful for understanding the mechanism's loads.

These are joint reaction forces, not stresses inside the rods. The rigid-rod model does not resolve internal bending or stress distributions.

In [ ]:
# Evaluate the four multipliers at every saved state.
lambda_func = hy.cfunc(list(lambda_expr), vars=q + qd)
lambda_inputs = np.ascontiguousarray(solution.T)
lambda_values = lambda_func(lambda_inputs).T

# Constraint order: top pin (x, y), then the hinge (x, y).
# The hinge reactions on the two rods are equal and opposite.
bar0_top_reaction = lambda_values[:, :2]
bar0_hinge_reaction = -lambda_values[:, 2:4]
bar1_hinge_reaction = lambda_values[:, 2:4]

bar0_top_magnitude = np.linalg.norm(bar0_top_reaction, axis=1)
bar0_hinge_magnitude = np.linalg.norm(bar0_hinge_reaction, axis=1)
bar1_hinge_magnitude = np.linalg.norm(bar1_hinge_reaction, axis=1)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)

axes[0].plot(tgrid, bar0_top_magnitude, label="Top pin")
axes[0].plot(tgrid, bar0_hinge_magnitude, label="Hinge")
axes[0].set_title("Rod 0 joint reactions")

axes[1].plot(tgrid, bar1_hinge_magnitude, label="Hinge")
axes[1].set_title("Rod 1 joint reactions")

for ax in axes:
    ax.set_xlabel("Time")
    ax.set_ylabel("Reaction-force magnitude")
    ax.grid(True)
    ax.legend()

plt.tight_layout()
plt.show()